# Step 6 — Xuất đặc trưng H3-B trên dữ liệu đã khóa (WSL)

Notebook này chạy các view và quy tắc chọn gate từ source `grace_h3v2` gốc, nhưng nạp trực tiếp raw split và CodeT5 NPZ theo `step6_lock.json` trong `GRACE-summary`. Trước khi fit, nó kiểm tra SHA-256 của raw split, row manifest, CodeT5 cache, thứ tự ID và donor bank đã khóa. Donor bank chỉ được ghi nhận/kiểm hash; đúng theo protocol, gate H3-B fit trên official train, không fit trên donor bank.

Gate chọn winner và ngưỡng từ nested OOF của train theo source H3-B; không dùng nhãn validation/test để fit hoặc chọn ngưỡng. `xpre_<split>.npz` có `X` 34 cột, `ids`, `feature_names`; `gate_<split>.csv` có `id,y,p_gate`. Dự đoán test được tạo từ code và embedding test; nhãn test chỉ được ghi vào CSV để evaluator ghép/đánh giá.

Script có cache model từng view và checkpoint từng outer fold để có thể resume nếu kernel/ngắt phiên. Đầu ra nằm trong `STEP6_supervised_anchors/runs/h3b_locked/`. Không có lệnh gọi LLM.

In [1]:
from pathlib import Path
import os, subprocess, sys

STEP6_ROOT = Path('/mnt/d/JUNIOR/LabThayTho/GRACE-summary/STEP6_supervised_anchors')
H3B_ROOT = Path('/mnt/d/JUNIOR/LabThayTho/GRACE-H3-BenchmarkEnsemble')
SCRIPT = STEP6_ROOT / 'src' / 'export_locked_h3b.py'
assert (STEP6_ROOT / 'step6_lock.json').is_file(), STEP6_ROOT
assert (H3B_ROOT / 'grace_h3v2' / 'features.py').is_file(), H3B_ROOT
assert SCRIPT.is_file(), SCRIPT
print('Step 6 root:', STEP6_ROOT)
print('H3-B source:', H3B_ROOT)
print('Python:', sys.executable)

Step 6 root: /mnt/d/JUNIOR/LabThayTho/GRACE-summary/STEP6_supervised_anchors
H3-B source: /mnt/d/JUNIOR/LabThayTho/GRACE-H3-BenchmarkEnsemble
Python: /mnt/d/JUNIOR/LabThayTho/GRACE-summary/.venv-wsl/bin/python


In [2]:
command = [sys.executable, str(SCRIPT), '--step6-root', str(STEP6_ROOT), '--h3b-root', str(H3B_ROOT)]
print('Running locked H3-B export; the first run fits nested base views and can take a while.')
proc = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
return_code = proc.wait()
if return_code != 0:
    raise RuntimeError(f'H3-B export failed with exit code {return_code}')

Running locked H3-B export; the first run fits nested base views and can take a while.
Preflight PASS: devign frozen raw/manifests, IDs and CodeT5 NPZ
Preflight PASS: reveal frozen raw/manifests, IDs and CodeT5 NPZ
devign: fit semantic_lr (14508 rows, seed=42)
devign: fit semantic_lr_balanced (14508 rows, seed=42)
devign: fit semantic_mlp (14508 rows, seed=42)
devign: fit lexical (14508 rows, seed=42)
devign: fit numeric (14508 rows, seed=42)
devign: fit semantic_lr (9672 rows, seed=30043)
devign: fit semantic_lr_balanced (9672 rows, seed=30043)
devign: fit semantic_mlp (9672 rows, seed=30043)
devign: fit lexical (9672 rows, seed=30043)
devign: fit numeric (9672 rows, seed=30043)
devign: fit semantic_lr (9672 rows, seed=31052)
devign: fit semantic_lr_balanced (9672 rows, seed=31052)
devign: fit semantic_mlp (9672 rows, seed=31052)
devign: fit lexical (9672 rows, seed=31052)
devign: fit numeric (9672 rows, seed=31052)
devign: fit semantic_lr (9672 rows, seed=32061)
devign: fit semantic_

In [3]:
import csv, hashlib, json
import numpy as np

lock_sha = hashlib.sha256((STEP6_ROOT / 'step6_lock.json').read_bytes()).hexdigest()
out_root = STEP6_ROOT / 'runs' / 'h3b_locked' / f'step6_lock_{lock_sha[:12]}'
for dataset in ('devign', 'reveal'):
    manifest_path = out_root / dataset / 'h3b_export_manifest.json'
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    print(dataset, 'winner=', manifest['h3b_source_algorithm']['winner'],
          'tau(train OOF)=', round(manifest['h3b_source_algorithm']['tau_train_oof'], 4))
    for split in ('validation', 'test'):
        xinfo = manifest['outputs'][split]['xpre']
        ginfo = manifest['outputs'][split]['gate']
        with np.load(out_root / xinfo['path'], allow_pickle=False) as z:
            print(' ', split, 'X=', z['X'].shape, 'ids=', len(z['ids']),
                  'gate_csv_rows=', ginfo['rows'], 'X_sha256=', xinfo['sha256'])

devign winner= fusion_all tau(train OOF)= 0.4
  validation X= (2732, 34) ids= 2732 gate_csv_rows= 2732 X_sha256= 44898c6090a9705d3e84d9ec9aefa0b0ce335766335886199976695ae6c37fa1
  test X= (2732, 34) ids= 2732 gate_csv_rows= 2732 X_sha256= 7122bbb11dc4819418447ea7bb280628533c1ca828ce6761c3d19a34b024b5b7
reveal winner= fusion_semantic_lexical tau(train OOF)= 0.17
  validation X= (2273, 34) ids= 2273 gate_csv_rows= 2273 X_sha256= fd88b1426aa6d2cd2cfcfbc6eb195f1b489194bf34bdbb2a486676601b0b60a8
  test X= (2274, 34) ids= 2274 gate_csv_rows= 2274 X_sha256= f5113d4e81d3b02cd9017a24214f19ab15af71df0bbd1a8802d9e47ec9c290f1
